In [ ]:
from datasets import load_dataset

open_subs = load_dataset("Helsinki-NLP/opus-100", "en-fr",cache_dir="/kaggle/working/hf_cache")
print(open_subs)



In [ ]:
open_subs.save_to_disk('/kaggle/working/opus100_enfr')

In [ ]:
from datasets import load_from_disk
ds = load_from_disk('/kaggle/working/opus100_enfr')

In [ ]:
pip install ftfy

In [ ]:
#main datacleaning script
import ftfy
import pandas as pd
def cleaning(dataset,aggressive:bool):
    #flattening the datatset to remove nested dicts
    flattened_ds = [v for k,i in dataset.items() for v in i]

    #removing whitespace from text and applying fix on text
    for i in flattened_ds:
          i['en'] = ftfy.fix_text(i['en'].strip())
          i['fr'] = ftfy.fix_text(i['fr'].strip())
    #removing empty pairs
    flattened_ds = [p for p in flattened_ds if p['en'] and p['fr']]
    if aggressive:
         #dropping misaligned translations
        flattened_ds = [
        p for p in flattened_ds
        if 0.5 <= len(p['en'].split()) / len(p['fr'].split()) <= 2.0]
    
        #removing long_sequence_pairs
        flattened_ds = [p for p in flattened_ds if len(p['en'].split())<=100 and len(p['fr'].split())<=100 ]
        df = pd.DataFrame(flattened_ds)
        flattened_ds = df.drop_duplicates().to_dict('records')
        #removing duplicates within  a translation
        flattened_ds = [p for p in flattened_ds if not (p['en'] == p['fr'] and len(p['en']) > 3)]
    return flattened_ds
    
   



In [ ]:
train_clean = cleaning(ds['train'][:], aggressive=True)
val_pairs = cleaning(ds['validation'][:], aggressive=False)
test_pairs = cleaning(ds['test'][:], aggressive=False)


In [ ]:
#creating cleaned dataset
from datasets import Dataset, DatasetDict
cleaned = DatasetDict({
    'train': Dataset.from_list(train_clean),
    'validation': Dataset.from_list(val_pairs),
    'test': Dataset.from_list(test_pairs),
})
cleaned.save_to_disk('/kaggle/working/opus100_cleaned')

In [ ]:
#loading cleaned dataset
from datasets import load_from_disk
cleaned = load_from_disk('/kaggle/working/opus100_cleaned')
print(cleaned)

In [ ]:
print(cleaned['train'][0:2])
print(cleaned['train']['fr'][0:2])

In [ ]:

#installing sentence piece
!pip install sentencepiece

In [ ]:
#writing sentences to txt
with open('/kaggle/working/sp_train.txt', 'w') as f:
    f.writelines(s + '\n' for s in cleaned['train']['en'])
    f.writelines(s + '\n' for s in cleaned['train']['fr'])

In [ ]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input='/kaggle/working/sp_train.txt',
    model_prefix='/kaggle/working/sp',
    vocab_size=32000,
    model_type='unigram',
    character_coverage=1.0,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    
)

In [ ]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor(model_file='/kaggle/working/sp.model')

# Basic info
print("vocab size:", sp.get_piece_size())
print("pad id:", sp.pad_id(), "unk id:", sp.unk_id(), "bos id:", sp.bos_id(), "eos id:", sp.eos_id())

# Encode a few examples
for sent in ["Hello, how are you?", "The cat sat on the mat.", "Bonjour, comment allez-vous?", "Le chat est noir."]:
    ids = sp.encode(sent, out_type=int)
    pieces = sp.encode(sent, out_type=str)
    print(f"\n{sent}")
    print(f"  pieces: {pieces}")
    print(f"  ids: {ids}")
    print(f"  decoded: {sp.decode(ids)}")

In [ ]:
print(sp.encode("Hello world", out_type=int, add_eos=True))
print(sp.encode("Bonjour le monde", out_type=int, add_bos=True, add_eos=True))

In [ ]:
#tokenization function

def tokenizer(ds_pair):
    en_ids = sp.encode(ds_pair['en'], out_type=int, add_eos=True)
    fr_ids = sp.encode(ds_pair['fr'], out_type=int, add_bos=True, add_eos=True)
    return {'en_ids': en_ids, 'fr_ids': fr_ids}
tokenized = cleaned.map(tokenizer,load_from_cache_file=False)
print(tokenized['train'][0])

In [ ]:
print(tokenized)

In [ ]:
#dropping the pure string keys

tokenized =tokenized.remove_columns({'en','fr'})


In [ ]:
#few sanity checks for train,test and val slpits

train_check = print(tokenized['train'][73:75])
test_check = print(tokenized['test'][73:75])
val_check = print(tokenized['validation'][73:75])

In [ ]:
#saving tokenization to disk
tokenized.save_to_disk('/kaggle/working/opus100_tokenized')

In [ ]:
# Sequence length stats — inform max_seq_len choice for the model later
import numpy as np
en_lens = [len(x) for x in tokenized['train']['en_ids']]
fr_lens = [len(x) for x in tokenized['train']['fr_ids']]
print(f"EN: mean={np.mean(en_lens):.1f}, median={np.median(en_lens)}, 95th={np.percentile(en_lens, 95):.0f}, 99th={np.percentile(en_lens, 99):.0f}, max={max(en_lens)}")
print(f"FR: mean={np.mean(fr_lens):.1f}, median={np.median(fr_lens)}, 95th={np.percentile(fr_lens, 95):.0f}, 99th={np.percentile(fr_lens, 99):.0f}, max={max(fr_lens)}")

In [ ]:
#removing ridicoulously long token squences 
def fits(ex):
    return len(ex['en_ids']) <= 128 and len(ex['fr_ids']) <= 128

tokenized = tokenized.filter(fits)

In [ ]:
tokenized.save_to_disk('/kaggle/working/opus100_tokenized')

In [ ]:
import torch as pt
device = pt.device('cuda' if pt.cuda.is_available() else 'cpu')

In [ ]:
class InputEmbedding(pt.nn.Module):
    def __init__(self,vocab_size,batch_size,embedding_dim):
        super().__init__()
        self.batch_size = batch_size
        self.embedding_dim = embedding_dim
        self.batch_list = []
        self.mask_list=[]
        self.embedded_batch = None
        self.dict_embedding = pt.nn.Embedding(num_embeddings=vocab_size,embedding_dim=embedding_dim).to(device)
        self.input_embedding =None
    def padding(self,input):
        self.batch_list =[]
        self.mask_list=[]
        for batch in range(0,len(input),self.batch_size):
            chunk = input[batch:batch+self.batch_size]
            padded = pt.nn.utils.rnn.pad_sequence(chunk, batch_first=True, padding_value=0).to(device)
            mask = (padded == 0)
            self.batch_list.append(padded)
            self.mask_list.append(mask)
        return self.batch_list,self.mask_list
            
    def embedding(self,batch_matrix):
        embeddings= self.dict_embedding
        self.embedded_batch = embeddings(batch_matrix)    
        return self.embedded_batch
    def positional_encoding(self,batch_embedding):
        postional_matrix = pt.zeros_like(batch_embedding).to(device)
        intermediate_calc = batch_embedding.shape[1]
        position_sin = pt.arange(intermediate_calc).unsqueeze(1).to(device)
        positions_cos = pt.arange(intermediate_calc).unsqueeze(1).to(device)
        intermediate_calc2= self.embedding_dim//2
        sin_embedding_dim = pt.arange(intermediate_calc2).to(device)
        cos_embedding_dim = pt.arange(intermediate_calc2).to(device)
        
        postional_matrix[:,:,0:intermediate_calc2]= pt.sin(position_sin/1000**((2*sin_embedding_dim)/self.embedding_dim)).to(device)
        postional_matrix[:,:,intermediate_calc2:]= pt.cos(positions_cos/1000**((2*cos_embedding_dim)/self.embedding_dim)).to(device)
        
        self.input_embedding = batch_embedding+postional_matrix
        
        return self.input_embedding

         
         
        
        
        

In [ ]:
#encoder logic

class MultiHeadAttention(pt.nn.Module):
    def __init__(self,heads,embedding_dim):
        super().__init__()
        self.heads = heads
        self.embedding = embedding_dim
        self.head_dim = self.embedding//self.heads
        self.query =pt.nn.Linear(in_features=embedding_dim , out_features=embedding_dim).to(device)
        self.key=pt.nn.Linear(in_features=embedding_dim , out_features=embedding_dim).to(device)
        self.value=pt.nn.Linear(in_features=embedding_dim , out_features=embedding_dim).to(device)
        self.projecion=pt.nn.Linear(in_features=embedding_dim , out_features=embedding_dim).to(device)
        
    def forward(self,input,src_mask):
        sequence_length = input.shape[1]
        batch_size = input.shape[0]
        query = self.query(input)
        self.q = pt.reshape(query,shape=(batch_size,sequence_length,self.heads,self.head_dim)).permute(0,2,1,3).to(device)
        key = self.key(input)
        self.k = pt.reshape(key,shape=(batch_size,sequence_length,self.heads,self.head_dim)).permute(0,2,1,3).to(device)
        value= self.value(input)
        self.v = pt.reshape(value,shape=(batch_size,sequence_length,self.heads,self.head_dim)).permute(0,2,1,3).to(device)
        
        attention_score = (pt.matmul(self.q,self.k.permute(0,1,3,2)))/(self.head_dim**0.5)
        attention_score = attention_score.masked_fill(src_mask[:, None, None, :], float('-inf'))
        attention_weights = pt.softmax(attention_score,dim=-1).to(device)
        context_vector = pt.matmul(attention_weights,self.v).permute(0, 2, 1, 3).contiguous().reshape(batch_size, sequence_length, self.embedding).to(device) #ai
        output= self.projecion(context_vector)
        return output
        
               
class LayerNorm(pt.nn.Module):
    def __init__(self,embedding_dim):
       super().__init__()
       self.gamma = pt.nn.Parameter(pt.ones(embedding_dim, device=device))
       self.beta = pt.nn.Parameter(pt.zeros(embedding_dim, device=device))
    def forward(self,input):
        mean = input.mean(dim=-1,keepdim=True)
        std = input.std(dim=-1,keepdim=True)
        normalization = (input-mean)/(std+1e-08)
        normalized_embedding= (normalization*self.gamma )+self.beta
        return normalized_embedding
    
    
class FeedForward(pt.nn.Module):
    def __init__(self,embedding_dim):
        super().__init__()
        self.layer1  = pt.nn.Linear(in_features=embedding_dim,out_features=4*embedding_dim).to(device)
        self.layer2 = pt.nn.Linear(in_features=4*embedding_dim,out_features=embedding_dim).to(device)
        
    def forward(self,input):
       first_layer= self.layer1(input)
       activation = pt.relu(first_layer).to(device)
       second_layer = self.layer2(activation)
       return second_layer
       

class EncoderBlock(pt.nn.Module):
    def __init__(self,embedding_dim,heads):
        super().__init__()
        self.selfattention = MultiHeadAttention(embedding_dim=embedding_dim,heads=heads)
        self.layernorm1 = LayerNorm(embedding_dim=embedding_dim)
        self.feedfoward = FeedForward(embedding_dim=embedding_dim)
        self.layernorm2 = LayerNorm(embedding_dim=embedding_dim)
        
    def forward(self,input,src_mask):
        attention_vectors = self.selfattention.forward(input,src_mask)
        add = input+attention_vectors
        normalization = self.layernorm1.forward(add)
        foward_output = self.feedfoward.forward(normalization)
        final_add = normalization+foward_output
        attention_output = self.layernorm2.forward(final_add)
        return attention_output
        
        
class Encoder(pt.nn.Module):
    def __init__(self,embedding_dim,heads,blocks):
        super().__init__()
        self.blocks = blocks
        self.encoderblock_list=pt.nn.ModuleList([EncoderBlock(embedding_dim=embedding_dim, heads=heads)
            for _ in range(blocks)
        ])
    def encoder_forward(self,input,src_mask):
        current = input
        for encoder in self.encoderblock_list:
            current = encoder.forward(current,src_mask)
        return current
            
enc = Encoder(embedding_dim=512, heads=8, blocks=6)
print(sum(p.numel() for p in enc.parameters()))

In [ ]:

class MaskedMultiHeadAttention(pt.nn.Module):
    def __init__(self, heads, embedding_dim):
        super().__init__()
        self.heads = heads
        self.embedding = embedding_dim
        self.head_dim = self.embedding // self.heads
        self.query = pt.nn.Linear(in_features=embedding_dim, out_features=embedding_dim)
        self.key = pt.nn.Linear(in_features=embedding_dim, out_features=embedding_dim)
        self.value = pt.nn.Linear(in_features=embedding_dim, out_features=embedding_dim)
        self.projection = pt.nn.Linear(in_features=embedding_dim, out_features=embedding_dim)

    def forward(self, input,trg_mask):
        sequence_length = input.shape[1]
        batch_size = input.shape[0]
        query = self.query(input)
        self.q = pt.reshape(query, shape=(batch_size, sequence_length, self.heads, self.head_dim)).permute(0, 2, 1, 3)
        key = self.key(input)
        self.k = pt.reshape(key, shape=(batch_size, sequence_length, self.heads, self.head_dim)).permute(0, 2, 1, 3)
        value = self.value(input)
        self.v = pt.reshape(value, shape=(batch_size, sequence_length, self.heads, self.head_dim)).permute(0, 2, 1, 3)

        attention_score = (pt.matmul(self.q, self.k.permute(0, 1, 3, 2))) / (self.head_dim ** 0.5)
        masked_matrix = pt.triu(pt.ones(sequence_length, sequence_length, dtype=pt.bool), diagonal=1).to(device)
        attention_score = attention_score.masked_fill(masked_matrix, float('-inf'))
        attention_score = attention_score.masked_fill(trg_mask[:, None, None, :], float('-inf'))
        attention_weights = pt.softmax(attention_score, dim=-1)
        context_vector = pt.matmul(attention_weights, self.v).permute(0, 2, 1, 3).contiguous().reshape(batch_size, sequence_length, self.embedding)
        output = self.projection(context_vector)
        return output


class LayerNorm(pt.nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.gamma = pt.nn.Parameter(pt.ones(embedding_dim))
        self.beta = pt.nn.Parameter(pt.zeros(embedding_dim))

    def forward(self, input):
        mean = input.mean(dim=-1, keepdim=True)
        std = input.std(dim=-1, keepdim=True)
        normalization = (input - mean) / (std + 1e-08)
        normalized_embedding = (normalization * self.gamma) + self.beta
        return normalized_embedding


class FeedFoward(pt.nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.layer1 = pt.nn.Linear(in_features=embedding_dim, out_features=4 * embedding_dim)
        self.layer2 = pt.nn.Linear(in_features=4 * embedding_dim, out_features=embedding_dim)

    def forward(self, input):
        first_layer = self.layer1(input)
        activation = pt.relu(first_layer)
        second_layer = self.layer2(activation)
        return second_layer


class MultiHeadCrossAttention(pt.nn.Module):
    def __init__(self, embedding_dim, heads):
        super().__init__()
        self.heads = heads
        self.embedding = embedding_dim
        self.head_dim = self.embedding // self.heads
        self.query = pt.nn.Linear(in_features=embedding_dim, out_features=embedding_dim)
        self.key = pt.nn.Linear(in_features=embedding_dim, out_features=embedding_dim)
        self.value = pt.nn.Linear(in_features=embedding_dim, out_features=embedding_dim)
        self.project = pt.nn.Linear(in_features=embedding_dim, out_features=embedding_dim)

    def forward(self, input_enc, input_dec,src_mask):
        sequence_length_dec = input_dec.shape[1]
        batch_size_dec = input_dec.shape[0]
        sequence_length_enc = input_enc.shape[1]
        batch_size_enc = input_enc.shape[0]
        query = self.query(input_dec)
        self.q = pt.reshape(query, shape=(batch_size_dec, sequence_length_dec, self.heads, self.head_dim)).permute(0, 2, 1, 3)
        key = self.key(input_enc)
        self.k = pt.reshape(key, shape=(batch_size_enc, sequence_length_enc, self.heads, self.head_dim)).permute(0, 2, 1, 3)
        value = self.value(input_enc)
        self.v = pt.reshape(value, shape=(batch_size_enc, sequence_length_enc, self.heads, self.head_dim)).permute(0, 2, 1, 3)
        attention_score = (pt.matmul(self.q, self.k.permute(0, 1, 3, 2))) / (self.head_dim ** 0.5)
        attention_score = attention_score.masked_fill(src_mask[:, None, None, :], float('-inf'))
        attention_weights = pt.softmax(attention_score, dim=-1)
        context_vector = pt.matmul(attention_weights, self.v).permute(0, 2, 1, 3).contiguous().reshape(batch_size_dec, sequence_length_dec, self.embedding)
        output = self.project(context_vector)
        return output


class DecoderBlock(pt.nn.Module):
    def __init__(self, heads, embedding_dim):
        super().__init__()
        self.maskedatttention = MaskedMultiHeadAttention(heads=heads, embedding_dim=embedding_dim)
        self.layernorm1 = LayerNorm(embedding_dim=embedding_dim)
        self.crossattention = MultiHeadCrossAttention(embedding_dim=embedding_dim, heads=heads)
        self.feedfoward = FeedFoward(embedding_dim=embedding_dim)
        self.layernorm2 = LayerNorm(embedding_dim=embedding_dim)
        self.layernorm3 = LayerNorm(embedding_dim=embedding_dim)

    def dec_block_forward(self, input, enc_output,src_mask,trg_mask):
        context_vector = self.maskedatttention(input,trg_mask)
        add = context_vector + input
        normalization = self.layernorm1(add)
        cross_attention = self.crossattention(enc_output, normalization,src_mask)
        add2 = cross_attention + normalization
        normalization2 = self.layernorm2(add2)
        linear = self.feedfoward(normalization2)
        add3 = linear + normalization2
        normalization3 = self.layernorm3(add3)
        return normalization3


class DECODER(pt.nn.Module):
    def __init__(self, embedding_dim, heads, blocks):
        super().__init__()
        self.decoderblock_list = pt.nn.ModuleList([
            DecoderBlock(embedding_dim=embedding_dim, heads=heads)
            for _ in range(blocks)
        ])

    def forward(self, input, enc_output,src_mask,trg_mask):
        current = input
        for decoder in self.decoderblock_list:
            current = decoder.dec_block_forward(current, enc_output,src_mask,trg_mask)
        return current

In [ ]:
import os
class Transformer(pt.nn.Module):
    def __init__(self, embedding_dim, heads, blocks, vocab_size_enc, vocab_size_dec, batch_size):
        super().__init__()
        self.encoder_embedding = InputEmbedding(vocab_size=vocab_size_enc, batch_size=batch_size, embedding_dim=embedding_dim)
        self.decoder_embedding = InputEmbedding(vocab_size=vocab_size_dec, batch_size=batch_size, embedding_dim=embedding_dim)
        self.Encoderblocks = Encoder(embedding_dim=embedding_dim, heads=heads, blocks=blocks)
        self.Decoderblocks = DECODER(embedding_dim=embedding_dim, heads=heads, blocks=blocks)
        self.linear = pt.nn.Linear(in_features=embedding_dim, out_features=vocab_size_dec)
        self.loss = pt.nn.CrossEntropyLoss(reduction='mean', ignore_index=0)

    def forward(self, source_batch, target_batch,src_mask,trg_mask):
        # encoder
        encoder_embed = self.encoder_embedding.embedding(source_batch)
        encoder_matrix = self.encoder_embedding.positional_encoding(encoder_embed)
        encoder_context_matrix = self.Encoderblocks.encoder_forward(encoder_matrix,src_mask)

        # decoder
        decoder_embed = self.decoder_embedding.embedding(target_batch)
        decoder_matrix = self.decoder_embedding.positional_encoding(decoder_embed)
        decoder_context_matrix = self.Decoderblocks(decoder_matrix, encoder_context_matrix,src_mask,trg_mask)

        # linear
        logits = self.linear(decoder_context_matrix)
        return logits

    def fit(self, input_enc, input_dec, epochs, learning_rate):
        encoder_padding,src_mask = self.encoder_embedding.padding(input_enc)
        decoder_padding,trg_mask = self.decoder_embedding.padding(input_dec)
        optimizer = pt.optim.Adam(self.parameters(), lr=learning_rate)
        scheduler = pt.optim.lr_scheduler.LambdaLR(
            optimizer,
            lambda step: min((step + 1) ** -0.5, (step + 1) * 4000 ** -1.5) * (4000 ** 0.5))
        loss_track = []
        global_step =0
        start_epoch=0
        scaler = pt.amp.GradScaler('cuda')
        if os.path.exists('/kaggle/working/checkpoint.pt'):
            ckpt = pt.load('/kaggle/working/checkpoint.pt')
            self.load_state_dict(ckpt['model'])
            optimizer.load_state_dict(ckpt['optimizer'])
            scheduler.load_state_dict(ckpt['scheduler'])
            scaler.load_state_dict(ckpt['scaler'])
            global_step = ckpt['step']
            start_epoch = ckpt['epoch']
            loss_track = ckpt['loss_track']
            print(f'Resumed from step {global_step}, epoch {start_epoch}')
            tmp_path = '/kaggle/working/checkpoint.pt.tmp'
            final_path = '/kaggle/working/checkpoint.pt'
        for epoch in range(start_epoch,epochs):
            batch_count = 0
            for batch_enc, batch_dec, batch_src_mask, batch_trg_mask in zip(encoder_padding, decoder_padding, src_mask, trg_mask):
                global_step+=1
                batch_count += 1
                if batch_count % 500 == 0:
                    print(f'batch: {batch_count}, loss: {loss.item():.4f}')

                batch_trg_mask = batch_trg_mask[:, :-1]  # match the decoder input shape
                decoder_input_batch = batch_dec[:, :-1]
                labels_batch = batch_dec[:, 1:]
                optimizer.zero_grad()
                with pt.amp.autocast('cuda'):
                    logits = self.forward(batch_enc, decoder_input_batch,batch_src_mask,batch_trg_mask)
                    logits = logits.permute(0, 2, 1)
                    loss = self.loss(logits, labels_batch)
                loss_track.append(loss.item())
                scaler.scale(loss).backward()  # scale loss, then backward
                scaler.step(optimizer)          # unscale grads, then optimizer.step()
                scaler.update()                 # adjust scale for next iteration
                scheduler.step()                # scheduler runs normally, separate from scaler
                wandb.log({
                        'loss': loss.item(),
                        'lr': scheduler.get_last_lr()[0],
                        'step': global_step,
                    })
                if global_step%750==0:
                    tmp_path = '/kaggle/working/checkpoint.pt.tmp'
                    final_path = '/kaggle/working/checkpoint.pt'
                    pt.save({
                            'step': global_step,
                            'epoch': epoch,
                            'model': self.state_dict(),
                            'optimizer': optimizer.state_dict(),
                            'scheduler': scheduler.state_dict(),
                            'scaler': scaler.state_dict(),
                            'loss_track': loss_track,
                        },tmp_path)
                    os.replace(tmp_path, final_path)
            print(f'epoch: {epoch}  loss: {loss.item():.4f}  lr: {scheduler.get_last_lr()[0]:.6f}')
        pt.save(self.state_dict(), 'transformer_weights.pt')
        return loss_track

In [ ]:
from datasets import load_from_disk
import torch as pt

device = pt.device('cuda')

tokenized = load_from_disk('/kaggle/working/opus100_tokenized')

train_en = [pt.tensor(x, dtype=pt.long,device=device) for x in tokenized['train']['en_ids']]
train_fr = [pt.tensor(x, dtype=pt.long,device=device) for x in tokenized['train']['fr_ids']]

# Trim to multiple of 64
n = (len(train_en) // 64) * 64
train_en = train_en[:n]
train_fr = train_fr[:n]

print(f'Training pairs: {len(train_en)}')

In [ ]:
import sentencepiece as spm
sp = spm.SentencePieceProcessor(model_file='/kaggle/working/sp.model')
pad_id = sp.pad_id()
print(f'Vocab size: {sp.get_piece_size()}, pad_id: {pad_id}')

In [ ]:
model = Transformer(
    embedding_dim=256,
    heads=4,
    blocks=4,
    vocab_size_enc=32000,
    vocab_size_dec=32000,
    batch_size=64
).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = "api key"

wandb.init(
    project='opus100-mt',
    name='second-translation',
    config={
        'embedding_dim': 256,
        'heads': 4,
        'blocks': 4,
        'vocab_size': 32000,
        'batch_size': 64,
        'learning_rate': 1e-4,
        'epochs': 22,
    }
)

In [ ]:
losses = model.fit(
    input_enc=train_en,
    input_dec=train_fr,
    epochs=22,
    learning_rate=1e-4
)

print('Training complete.')
print(f'Final loss: {losses[-1]:.4f}')

In [ ]:
#inference section
#use a fubction that takes in an english word and returns at the end a french string

def translate(model,sp,english_sentence,max_len=128,device=device):
    #tokenize input
     english_tokens=sp.encode(english_sentence,out_type=int,add_eos=True)
     english_tokens = pt.tensor(english_tokens, dtype=pt.long, device=device).unsqueeze(0)
     print(english_tokens.shape)
    # adding a false mask because the tranformer expects some masking
     src_mask = pt.zeros(1, english_tokens.shape[1], dtype=pt.bool, device=device)
     enc_embed = model.encoder_embedding.embedding(english_tokens)
     enc_embed = model.encoder_embedding.positional_encoding(enc_embed)
     enc_out = model.Encoderblocks.encoder_forward(enc_embed, src_mask)
     print(enc_out.shape)

     #decoder section
     bos_id = sp.bos_id()
     eos_id = sp.eos_id()
     decoder_input = pt.tensor([[bos_id]], dtype=pt.long, device=device)
     print(bos_id,eos_id,decoder_input)
     for _ in range(max_len):
         trg_mask = pt.zeros(1, decoder_input.shape[1], dtype=pt.bool, device=device)
         dec_embed = model.decoder_embedding.embedding(decoder_input)
         dec_embed = model.decoder_embedding.positional_encoding(dec_embed)
         dec_out = model.Decoderblocks(dec_embed,enc_out,src_mask,trg_mask)
         logits = model.linear(dec_out)
         logits =logits[:, -1, :]
         next_token = pt.argmax(logits ,-1, keepdim=True)
         decoder_input = pt.cat([decoder_input,next_token],dim=1)
         if next_token == eos_id:
             break
     generated_ids = decoder_input.squeeze(0).tolist()[1:]  # skip <s>
     french = sp.decode(generated_ids)
     return french
    
sp =spm.SentencePieceProcessor(model_file='/kaggle/working/sp.model')
model = Transformer(    embedding_dim=256, heads=4, blocks=4,
    vocab_size_enc=32000, vocab_size_dec=32000,
    batch_size=64).to(device)

print(translate(model,sp,"hey how are you"))
